# Variables climáticas mensuales desde estaciones del IDEAM

**Proyecto:** pronóstico de la volatilidad de precios de arroz y carne de res en Colombia (SIPSA-DANE).

Este notebook construye variables climáticas **mensuales** de las zonas productoras a partir de datos
**diarios** de estaciones del IDEAM, y las deja listas para unirse con las series de precios.

**Alcance:** solo se usa información del IDEAM. No se rellena ni se mezcla con fuentes satelitales; los meses
sin dato suficiente quedan como `NaN` y se reportan.

**Flujo:**
1. Configuración (periodos, criterios, pesos)
2. Descarga de insumos (archivos masivos del IDEAM y Catálogo Nacional de Estaciones)
3. Lectura de archivos y catálogo
4. Control de calidad diario
5. Indicadores mensuales por estación (con la explicación de cada uno)
6. Selección de estaciones
7. Agregación por departamento y por producto
8. Ejecución y salidas (`salidas/`)

**Uso:** ejecutar el notebook desde la carpeta `raw_data/`. Los insumos se guardan en `datos_ideam/` (excluida de git);
si ya existen, no se vuelven a descargar.

## 1. Configuración

**Periodos**
- Estudio: enero de 2013 a octubre de 2026. Los masivos del IDEAM se publicaron el 13-14 de agosto de 2026, por lo que
  agosto-octubre de 2026 quedan `NaN` hasta que el IDEAM los actualice (basta borrar los zip y volver a ejecutar).
- Periodo base 1991-2020 (normal climatológica de la OMM): solo se usa para calcular el percentil 90 de Tmax.
  Todo lo demás fuera del periodo de estudio se descarta al leer.

**Selección de estaciones**
- Departamentos de las zonas productoras, por debajo de 1.000 m s. n. m. (tierra caliente).
- Al menos 70 % de días con dato en el periodo de estudio, evaluado por variable (una estación puede servir para
  lluvia y no para temperatura).
- Para `dias_calor_p90`: además, al menos 10 años de Tmax entre 1991 y 2012.
- No se excluye ninguna tecnología: las series `_CON` son las lecturas convencionales aunque la estación hoy sea
  mixta o automática (ver sección 3).

**Pesos por departamento** (fijos, promedio del periodo, para no introducir cambios artificiales; solo importan las
proporciones):
- **Arroz:** producción anual de arroz mecanizado (miles de t), promedio 2013-2025. Fuente: DANE-Fedearroz, ENAM,
  anexo del I semestre 2026, hoja *Históricos*. Norte de Santander se publica aparte solo desde 2018 (promedio 2018-2025).
  Se prefiere producción a área sembrada porque el precio responde a la oferta (Tolima, con riego, rinde más por hectárea).
- **Res:** inventario bovino (millones de cabezas), promedio de los censos del ICA 2018 y 2020-2025
  (el archivo de 2019 está cifrado).

| Producto | Departamento | Peso | % |
|---|---|---|---|
| Arroz | Casanare | 881,6 | 35,9 |
| Arroz | Tolima | 713,3 | 29,0 |
| Arroz | Meta | 387,4 | 15,8 |
| Arroz | Huila | 239,5 | 9,7 |
| Arroz | Norte de Santander | 235,1 | 9,6 |
| Res | Antioquia | 3,193 | 23,3 |
| Res | Córdoba | 2,310 | 16,9 |
| Res | Meta | 2,241 | 16,4 |
| Res | Casanare | 2,200 | 16,1 |
| Res | Caquetá | 2,154 | 15,7 |
| Res | Cesar | 1,583 | 11,6 |

In [1]:
from __future__ import annotations

import re
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# --- Periodos ----------------------------------------------------------------
FECHA_INICIO = "2013-01-01"     # inicio del periodo de estudio
FECHA_FIN = "2026-10-31"        # fin del periodo de estudio (masivos publicados el 13-14 ago 2026)
BASE_INICIO = "1991-01-01"      # periodo base (normal OMM) para el percentil 90
BASE_FIN = "2020-12-31"

# --- Selección de estaciones ---------------------------------------------------
ALTITUD_MAX = 1000              # m s. n. m. (tierra caliente)
ALTITUDES_REPORTE = [1000, 1500]  # se reporta cuántas estaciones quedarían con cada altitud
COMPLETITUD_MIN = 0.70          # fracción mínima de días con dato en el periodo de estudio
ANIOS_BASE_MIN = 10             # años mínimos de Tmax entre BASE_INICIO y FECHA_INICIO (para p90)
TECNOLOGIAS_EXCLUIDAS: list[str] = []   # p. ej. ["Automática con Telemetría"] para ser conservador

# --- Control de calidad ----------------------------------------------------------
PRECIP_MIN, PRECIP_MAX = 0.0, 500.0     # mm/día
TMAX_MIN, TMAX_MAX = 5.0, 45.0          # °C
DIAS_MES_MIN = 0.80                     # fracción mínima de días con dato para que el mes valga

# --- Indicadores -------------------------------------------------------------------
UMBRAL_LLUVIA_MM = 1.0          # día con lluvia (convención OMM)
UMBRAL_LLUVIA_FUERTE_MM = 20.0  # lluvia fuerte (encharcamiento / inundación)
UMBRAL_TMAX_ARROZ = 35.0        # °C, estrés térmico del arroz en floración
PERCENTIL_CALOR = 0.90

# --- Agregación --------------------------------------------------------------------
# Un mes-producto queda NaN si los departamentos con dato suman menos de esta fracción del peso.
PESO_MIN_PRODUCTO = 0.50

PESOS = {
    "arroz": {"Casanare": 881.6, "Meta": 387.4, "Tolima": 713.3, "Huila": 239.5, "Norte de Santander": 235.1},
    "res": {"Antioquia": 3.193, "Meta": 2.241, "Casanare": 2.200, "Córdoba": 2.310, "Caquetá": 2.154, "Cesar": 1.583},
}

# Variables de la tabla del modelo (sin THI: el IDEAM no tiene humedad diaria completa en estas estaciones)
VARIABLES_MODELO = {
    "arroz": ["tmax_media", "dias_calor_p90", "dias_tmax_ge35",
              "precip_total_mm", "dias_lluvia", "dias_lluvia_fuerte", "max_racha_seca"],
    "res": ["tmax_media", "dias_calor_p90",
            "precip_total_mm", "dias_lluvia", "dias_lluvia_fuerte", "max_racha_seca"],
}

# --- Rutas y fuentes -------------------------------------------------------------
CARPETA_DATOS = Path("datos_ideam")     # insumos crudos (no se suben a git)
CARPETA_SALIDA = Path("salidas")

URLS_IDEAM = {
    "PrecipitacionNacionalDiaria.zip": "https://bart.ideam.gov.co/PQRS/AQTSUtils/PrecipitacionNacionalDiaria.zip",
    "TemperaturaNacionalDiaria.zip": "https://bart.ideam.gov.co/PQRS/AQTSUtils/TemperaturaNacionalDiaria.zip",
}
URL_CNE = "https://www.datos.gov.co/resource/hp9r-jxuu.csv?$limit=100000"   # CNE del IDEAM en datos.gov.co
ARCHIVO_CNE = "CNE_IDEAM.csv"

ET_PRECIP, ET_TMAX, ET_TMIN = "PTPM_CON_INTER", "TSSM_MX_D", "TSSM_MN_D"

VARIABLES = ["tmax_media", "tmin_media", "dias_calor_p90", "dias_tmax_ge35",
             "precip_total_mm", "dias_lluvia", "dias_lluvia_fuerte", "max_racha_seca"]

MESES = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="MS")
DIAS = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D")

## 2. Descarga de insumos

- **Archivos masivos del IDEAM** (`bart.ideam.gov.co`): precipitación diaria (~139 MB) y temperatura diaria (~117 MB).
  El enlace `http://` publicado no responde (puerto 80 cerrado); el **mismo archivo del mismo servidor** se descarga por
  `https://` con certificado válido.
- **Catálogo Nacional de Estaciones (CNE)** del IDEAM, publicado en datos.gov.co (conjunto `hp9r-jxuu`).

La descarga es por *streaming*, muestra el progreso y se omite si el archivo ya existe. Los zip **no se descomprimen**
(ocupan ~1,8 GB y más de 7.000 archivos): se leen directamente solo los archivos de las estaciones elegidas.

In [2]:
def descargar_streaming(url: str, destino: Path, bloque: int = 1 << 20) -> None:
    """Descarga por streaming con barra de progreso. No re-descarga si el archivo ya existe."""
    if destino.exists() and destino.stat().st_size > 0:
        print(f"  ya existe: {destino} ({destino.stat().st_size / 1e6:,.1f} MB) -> se omite")
        return
    tmp = destino.with_suffix(destino.suffix + ".part")
    print(f"  descargando {url}")
    with requests.get(url, stream=True, timeout=(30, 300)) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length") or 0)
        leido = 0
        with open(tmp, "wb") as f:
            for trozo in r.iter_content(bloque):
                f.write(trozo)
                leido += len(trozo)
                if total:
                    print(f"\r    {leido / 1e6:8.1f} / {total / 1e6:.1f} MB ({100 * leido / total:5.1f} %)",
                          end="", flush=True)
    print()
    if total and leido != total:
        raise RuntimeError(f"Descarga incompleta de {url}: {leido} de {total} bytes")
    tmp.replace(destino)


def descargar_insumos() -> None:
    CARPETA_DATOS.mkdir(exist_ok=True)
    for nombre, url in URLS_IDEAM.items():
        descargar_streaming(url, CARPETA_DATOS / nombre)
    descargar_streaming(URL_CNE, CARPETA_DATOS / ARCHIVO_CNE)


descargar_insumos()

  ya existe: datos_ideam\PrecipitacionNacionalDiaria.zip (139.3 MB) -> se omite
  ya existe: datos_ideam\TemperaturaNacionalDiaria.zip (117.4 MB) -> se omite
  ya existe: datos_ideam\CNE_IDEAM.csv (6.9 MB) -> se omite


## 3. Archivos, etiquetas y catálogo de estaciones

### Formato de los archivos `.data`
Cada zip contiene archivos `ETIQUETA@CODIGO.data` (por ejemplo `PTPM_CON_INTER@11010010.data`), con encabezado
`Fecha|Valor`, separador `|`, fecha `AAAA-MM-DD HH:MM:SS`, decimal con punto y sin columna de calidad.

### Etiquetas (verificadas con los datos)
| Etiqueta | Qué es |
|---|---|
| `PTPM_CON_INTER` | Precipitación total diaria: un valor por día con hora 07:00 (lluvia acumulada en las 24 h previas). Se asigna a la fecha de lectura, como la publica el IDEAM. |
| `TSSM_MX_D` | **Máximo de las 3 lecturas diarias** de termómetro seco (07, 13 y 19 h). Coincide en el 100 % de los días con el máximo de `TSSM_CON`. Aproxima la Tmax. |
| `TSSM_MN_D` | Mínimo de esas 3 lecturas. Aproxima la Tmin. |

**Importante:** los masivos del IDEAM **no incluyen** la máxima/mínima de termómetro (`TMX_CON`/`TMN_CON`).
Por eso la Tmax usada subestima la real (el pico ocurre hacia las 14-15 h, no a las 13 h) y la Tmin la sobreestima
(el mínimo ocurre antes de las 07 h). El sesgo es aproximadamente constante en cada estación.

### Unión con el CNE
- El código del archivo (8 dígitos) se une al `codigo` del CNE (10 dígitos con ceros a la izquierda) como número entero.
- El CNE de datos.gov.co trae **cada fila dos veces**: se eliminan duplicados.
- La columna `ubicaci_n` trae latitud y longitud **invertidas**: se usan las columnas `latitud` y `longitud`.
- Los departamentos vienen sin tilde (`Cordoba`, `Caqueta`, `Norte De Santander`): se comparan sin tildes ni mayúsculas.

### Estaciones convencionales, mixtas y automáticas
La columna `tecnologia` del CNE describe **cómo mide hoy la estación**:
- **Convencional:** un observador lee los instrumentos a mano (pluviómetro a las 07:00, termómetros a las 07, 13 y 19 h).
  Es el dato oficial validado.
- **Automática:** sensores electrónicos que registran cada pocos minutos y transmiten por telemetría. El IDEAM advierte que
  estos datos son **preliminares** (sin validación técnica, Decreto 1076 de 2015).
- **Mixta** (*Automática con Telemetría, Convencional*): el mismo sitio tiene ambos sistemas.

Las etiquetas usadas son de la familia `_CON` (serie **convencional**): en una estación mixta el archivo contiene las
lecturas del observador, no las del sensor. Por eso se incluyen todas (`TECNOLOGIAS_EXCLUIDAS` permite excluirlas).

In [3]:
def normalizar(txt: str) -> str:
    """Quita tildes y pasa a minúsculas (el CNE escribe 'Cordoba', 'Norte De Santander')."""
    txt = unicodedata.normalize("NFKD", str(txt)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", txt).strip().lower()


def cargar_cne() -> pd.DataFrame:
    cne = pd.read_csv(CARPETA_DATOS / ARCHIVO_CNE, dtype=str)
    cne = cne.drop_duplicates("codigo")          # cada fila viene dos veces
    cne["cod_int"] = cne["codigo"].astype(int)   # archivos: 8 dígitos; CNE: 10 con ceros a la izquierda
    for c in ("altitud", "latitud", "longitud"):  # 'ubicaci_n' trae lat/lon invertidas: no se usa
        cne[c] = pd.to_numeric(cne[c], errors="coerce")
    deptos = {normalizar(d): d for pesos in PESOS.values() for d in pesos}
    cne["depto"] = cne["departamento"].map(lambda d: deptos.get(normalizar(d)))
    return cne


def indice_zip(z: zipfile.ZipFile) -> dict[str, dict[int, str]]:
    """{etiqueta: {codigo: nombre_archivo}} a partir de los nombres ETIQUETA@CODIGO.data."""
    out: dict[str, dict[int, str]] = {}
    for nombre in z.namelist():
        m = re.match(r"(.+)@(\d+)\.data$", nombre.split("/")[-1])
        if m:
            out.setdefault(m.group(1), {})[int(m.group(2))] = nombre
    return out


def leer_data(z: zipfile.ZipFile, nombre: str, desde: str, hasta: str) -> tuple[pd.Series, int]:
    """Lee un .data, descarta lo que está fuera de [desde, hasta] y devuelve la serie diaria
    (índice = fecha de lectura) y el número de días repetidos descartados."""
    d = pd.read_csv(z.open(nombre), sep="|", dtype={"Fecha": str, "Valor": float})
    d["dia"] = d["Fecha"].str[:10]
    d = d[(d["dia"] >= desde) & (d["dia"] <= hasta)]
    n_dup = int(d["dia"].duplicated().sum())
    d = d.drop_duplicates("dia", keep="first")
    s = pd.Series(d["Valor"].to_numpy(), index=pd.to_datetime(d["dia"], format="%Y-%m-%d"))
    return s.sort_index(), n_dup


cne = cargar_cne()
zp = zipfile.ZipFile(CARPETA_DATOS / "PrecipitacionNacionalDiaria.zip")
zt = zipfile.ZipFile(CARPETA_DATOS / "TemperaturaNacionalDiaria.zip")
ip, it = indice_zip(zp), indice_zip(zt)
print(f"CNE: {len(cne)} estaciones únicas")
for et, d in {**ip, **it}.items():
    print(f"  {et}: {len(d)} archivos")

CNE: 9930 estaciones únicas
  PTPM_CON_INTER: 4450 archivos
  TSSM_CON: 652 archivos
  TSSM_MEDIA_D: 647 archivos
  TSSM_MN_D: 631 archivos
  TSSM_MX_D: 632 archivos


## 4. Control de calidad diario

Se descartan valores imposibles:
- Precipitación < 0 o > 500 mm/día.
- Tmax < 5 °C o > 45 °C.
- Tmin > Tmax (se descartan ambos valores del día).
- Fechas repetidas (se conserva la primera).

No se descartan días por número de lecturas de termómetro seco: en la práctica el IDEAM solo calcula `TSSM_MX_D`
cuando el día tiene las 3 lecturas (solo 79 de más de 1,7 millones de días tenían menos).

Luego, por estación y mes, si hay **menos del 80 % de días con dato**, ese mes queda `NaN` para esa estación.
El detalle de lo descartado queda en la hoja `calidad`.

In [4]:
def diario_precip(codigo) -> tuple[pd.Series, dict]:
    s, n_dup = leer_data(zp, ip[ET_PRECIP][codigo], FECHA_INICIO, FECHA_FIN)
    malos = (s < PRECIP_MIN) | (s > PRECIP_MAX)
    q = {"precip_leidos": int(s.notna().sum()), "precip_dias_repetidos": n_dup,
         "precip_descartados_rango": int(malos.sum())}
    return s.mask(malos), q


def diario_temp(codigo) -> tuple[pd.DataFrame, dict]:
    """Tmax/Tmin diarias desde BASE_INICIO (la base se necesita para el percentil 90)."""
    desde = min(BASE_INICIO, FECHA_INICIO)
    tmax, dup_x = leer_data(zt, it[ET_TMAX][codigo], desde, FECHA_FIN)
    tmin, dup_n = (leer_data(zt, it[ET_TMIN][codigo], desde, FECHA_FIN)
                   if codigo in it.get(ET_TMIN, {}) else (pd.Series(dtype=float), 0))
    d = pd.DataFrame({"tmax": tmax, "tmin": tmin})
    q = {"temp_leidos": int(d["tmax"].notna().sum()), "temp_dias_repetidos": dup_x + dup_n}
    fuera = (d["tmax"] > TMAX_MAX) | (d["tmax"] < TMAX_MIN)
    q["tmax_descartados_rango"] = int(fuera.sum())
    d.loc[fuera, "tmax"] = np.nan
    invertida = d["tmin"] > d["tmax"]
    q["temp_descartados_tmin_gt_tmax"] = int(invertida.sum())
    d.loc[invertida, ["tmax", "tmin"]] = np.nan
    return d, q

## 5. Indicadores mensuales por estación

Todos se calculan por estación y mes. Los indicadores que son **conteos de días** se ajustan al largo del mes
cuando faltan días: `conteo × días_del_mes / días_válidos`. Por ejemplo, si marzo tiene 28 días con dato y 3 cumplen
la condición, se reporta 3 × 31 / 28 = 3,3. Las columnas `reescalado_*` indican cuándo se hizo.

### Temperatura
| Variable | Cálculo |
|---|---|
| `tmax_media` | Promedio mensual de la Tmax diaria. |
| `tmin_media` | Promedio mensual de la Tmin diaria (se calcula, pero no entra a la tabla del modelo). |
| `dias_tmax_ge35` | Días del mes con Tmax ≥ 35 °C (umbral **fijo**). |
| `dias_calor_p90` | Días del mes con Tmax mayor que el percentil 90 de **esa estación** en ese mes calendario (umbral **relativo**). |

Los dos indicadores de calor cuentan **cuántos días calurosos hubo en el mes**. La diferencia está en qué se considera
"caluroso".

**`dias_tmax_ge35` — umbral fijo.** Se cuentan los días con Tmax ≥ 35 °C. Es el umbral de estrés térmico del arroz:
por encima de unos 35 °C durante la floración el polen pierde viabilidad, aumenta la esterilidad de las espiguillas y
cae el rendimiento. Es un umbral **fisiológico**, igual en cualquier lugar. Limitación: en zonas menos calientes casi
siempre da 0 y, con nuestra Tmax aproximada (máximo de las lecturas de 07, 13 y 19 h), **cuenta menos días de los
reales**. Debe leerse como cota inferior.

**`dias_calor_p90` — umbral relativo a cada estación.**
1. *Umbral* (una vez por estación): se toman todas las Tmax de un mismo mes calendario en 1991-2020 (por ejemplo,
   todos los días de marzo de esos 30 años) y se calcula el valor que solo supera el 10 % de esos días: el percentil 90.
   Resultan 12 umbrales por estación, uno por mes.
2. *Conteo* (cada mes): se cuentan los días con Tmax por encima del umbral de su mes.

Mide calor **anómalo para ese lugar y esa época del año**. Un mes "normal" tendría unos 3 días (10 %); un mes con 9 o
más fue inusualmente caluroso. Es el índice TX90p de la OMM (ETCCDI), responde bien a El Niño y **no lo afecta el sesgo
de la Tmax aproximada**, porque el umbral y los valores diarios salen de la misma medición. Es el indicador principal
de calor.

En el periodo de estudio promedia unos 4 días por mes, no 3. Se verificó que no es un error: en el conjunto del
periodo base el 8,9 % de los días supera el p90 (menos del 10 % por empates, porque la Tmax se registra cada 0,2 °C),
pero la frecuencia crece en el tiempo: 6,6 % en 1991-2002, 7,5 % en 2003-2012 y 13,5 % en 2013-2020. Es la tendencia
de calentamiento (y El Niño 2015-16). En la sección 8 hay un ejemplo con una estación real.

### Precipitación
| Variable | Cálculo |
|---|---|
| `precip_total_mm` | Suma de la lluvia diaria del mes (no se reescala; exige al menos 80 % de días con dato). |
| `dias_lluvia` | Días con lluvia ≥ 1 mm (convención de la OMM para "día con lluvia"). |
| `dias_lluvia_fuerte` | Días con lluvia ≥ 20 mm (riesgo de encharcamiento, inundación y daño de vías). |
| `max_racha_seca` | Máximo de días **consecutivos** con lluvia < 1 mm dentro del mes. Es un máximo, no un conteo: no se reescala, y los días faltantes cortan la racha. |

### ¿Y lo contrario: lluvia excesiva y frío?
- **Exceso de lluvia** sí está medido, con umbrales fijos: `precip_total_mm` (cantidad), `dias_lluvia` (frecuencia) y
  `dias_lluvia_fuerte` (intensidad). Su opuesto, la sequía, lo mide `max_racha_seca`, junto con valores bajos de los
  anteriores. A diferencia del calor, la lluvia **no tiene un indicador relativo** tipo percentil.
- **Frío** no tiene un indicador de extremos: solo se calcula `tmin_media`, que no entra a la tabla del modelo. En
  tierra caliente (< 1.000 m) el frío extremo es poco frecuente y el riesgo climático dominante para arroz y ganado es
  el calor y la falta o el exceso de agua. Las noches frescas sí pueden afectar al arroz (frío en la etapa de
  embuchamiento), pero eso ocurre sobre todo en zonas más altas.

In [5]:
def _racha_seca_mensual(precip: pd.Series) -> pd.Series:
    """Máximo de días consecutivos con lluvia < 1 mm dentro de cada mes (los NaN cortan la racha)."""
    seco = (precip < UMBRAL_LLUVIA_MM) & precip.notna()
    mes_idx = precip.index.to_period("M")
    mes = pd.Series(mes_idx, index=precip.index)
    corte = (~seco) | (mes != mes.shift())          # nueva racha si no es seco o si cambia el mes
    largo = seco.astype(int).groupby(corte.cumsum()).cumsum()
    out = largo.groupby(mes_idx).max().astype(float)
    out.index = out.index.to_timestamp()
    return out


def _conteo(cond: pd.Series, valido: pd.Series, n_val: pd.Series, dias_mes: pd.Series) -> pd.Series:
    """Conteo mensual de días que cumplen la condición, reescalado al largo del mes."""
    c = (cond & valido).astype(float).resample("MS").sum().reindex(MESES)
    return c * dias_mes / n_val


def mensual_precip(p: pd.Series) -> pd.DataFrame:
    p = p.reindex(DIAS)
    ok = p.notna()
    dias_mes = pd.Series(MESES.days_in_month, index=MESES, dtype=float)
    n_val = ok.astype(int).resample("MS").sum().reindex(MESES)
    valido_mes = n_val >= DIAS_MES_MIN * dias_mes
    out = pd.DataFrame({
        "precip_total_mm": p.resample("MS").sum(min_count=1).reindex(MESES),
        "dias_lluvia": _conteo(p >= UMBRAL_LLUVIA_MM, ok, n_val, dias_mes),
        "dias_lluvia_fuerte": _conteo(p >= UMBRAL_LLUVIA_FUERTE_MM, ok, n_val, dias_mes),
        "max_racha_seca": _racha_seca_mensual(p).reindex(MESES),
    })
    out = out.where(valido_mes)
    out["dias_validos_lluvia"] = n_val
    out["reescalado_lluvia"] = valido_mes & (n_val < dias_mes)
    return out


def umbrales_p90(d: pd.DataFrame) -> pd.Series:
    """Percentil 90 de Tmax por mes calendario en el periodo base de la propia estación."""
    base = d.loc[BASE_INICIO:BASE_FIN, "tmax"].dropna()
    return base.groupby(base.index.month).quantile(PERCENTIL_CALOR)


def mensual_temp(d: pd.DataFrame, usar_p90: bool) -> pd.DataFrame:
    p90 = umbrales_p90(d)
    d = d.reindex(DIAS)
    ok_x, ok_n = d["tmax"].notna(), d["tmin"].notna()
    dias_mes = pd.Series(MESES.days_in_month, index=MESES, dtype=float)
    n_x = ok_x.astype(int).resample("MS").sum().reindex(MESES)
    n_n = ok_n.astype(int).resample("MS").sum().reindex(MESES)
    valido_x = n_x >= DIAS_MES_MIN * dias_mes
    valido_n = n_n >= DIAS_MES_MIN * dias_mes
    umbral = pd.Series(d.index.month, index=d.index).map(p90)
    out = pd.DataFrame({
        "tmax_media": d["tmax"].resample("MS").mean().reindex(MESES).where(valido_x),
        "tmin_media": d["tmin"].resample("MS").mean().reindex(MESES).where(valido_n),
        "dias_calor_p90": _conteo(d["tmax"] > umbral, ok_x, n_x, dias_mes).where(valido_x),
        "dias_tmax_ge35": _conteo(d["tmax"] >= UMBRAL_TMAX_ARROZ, ok_x, n_x, dias_mes).where(valido_x),
    })
    if not usar_p90:
        out["dias_calor_p90"] = np.nan
    out["dias_validos_temp"] = n_x
    out["reescalado_temp"] = valido_x & (n_x < dias_mes)
    return out

## 6. Selección de estaciones y cálculo por estación

1. Filtro previo con el CNE: departamentos objetivo, altitud (se leen hasta 1.500 m para poder reportar ambos
   escenarios) y tecnología. Solo se leen del zip los archivos de esas estaciones.
2. Para cada estación se calcula la completitud de cada variable en el periodo de estudio y los años de Tmax en la base.
3. Se usa la estación para una variable si está por debajo de `ALTITUD_MAX` y cumple el 70 % de completitud; para
   `dias_calor_p90` además debe tener al menos 10 años de base.

In [6]:
def procesar_estaciones():
    cand = cne[cne["depto"].notna()
               & (cne["altitud"] <= max(ALTITUDES_REPORTE + [ALTITUD_MAX]))
               & ~cne["tecnologia"].isin(TECNOLOGIAS_EXCLUIDAS)]
    cand = cand[cand["cod_int"].isin(set(ip.get(ET_PRECIP, {})) | set(it.get(ET_TMAX, {})))]
    print(f"Estaciones candidatas: {len(cand)}")

    fichas, calidad, mens = [], [], {}
    n_dias = len(DIAS)
    for i, r in enumerate(cand.itertuples(), 1):
        if i % 200 == 0:
            print(f"  {i}/{len(cand)}")
        c = r.cod_int
        ficha = {"codigo": r.codigo, "nombre": r.nombre, "departamento": r.depto, "municipio": r.municipio,
                 "latitud": r.latitud, "longitud": r.longitud, "altitud": r.altitud,
                 "categoria": r.categoria, "tecnologia": r.tecnologia, "estado": r.estado}
        q = {"codigo": r.codigo, "departamento": r.depto}
        precip = temp = None
        if c in ip.get(ET_PRECIP, {}):
            precip, qp = diario_precip(c)
            q.update(qp)
            ficha["completitud_precip"] = precip.reindex(DIAS).notna().sum() / n_dias
        if c in it.get(ET_TMAX, {}):
            temp, qt = diario_temp(c)
            q.update(qt)
            ficha["completitud_temp"] = temp["tmax"].reindex(DIAS).notna().sum() / n_dias
            pre = temp.loc[BASE_INICIO:pd.Timestamp(FECHA_INICIO) - pd.Timedelta(days=1), "tmax"]
            ficha["anios_base_tmax"] = pre.notna().sum() / 365.25

        en_alt = r.altitud <= ALTITUD_MAX
        usa_p = en_alt and ficha.get("completitud_precip", 0) >= COMPLETITUD_MIN
        usa_t = en_alt and ficha.get("completitud_temp", 0) >= COMPLETITUD_MIN
        usa_p90 = usa_t and ficha.get("anios_base_tmax", 0) >= ANIOS_BASE_MIN
        ficha.update(usada_precip=usa_p, usada_temp=usa_t, usada_p90=usa_p90)
        fichas.append(ficha)

        m = []
        if usa_p:
            m.append(mensual_precip(precip))
        if usa_t:
            m.append(mensual_temp(temp, usa_p90))
        if m:
            mm = pd.concat(m, axis=1)
            mens[r.codigo] = mm
            if usa_p:
                q["meses_nan_lluvia"] = int(mm["precip_total_mm"].isna().sum())
            if usa_t:
                q["meses_nan_temp"] = int(mm["tmax_media"].isna().sum())
        calidad.append(q)

    fichas = pd.DataFrame(fichas)
    for c in ("completitud_precip", "completitud_temp", "anios_base_tmax"):
        if c not in fichas:
            fichas[c] = np.nan
    return fichas, pd.DataFrame(calidad), mens


fichas, calidad, mens = procesar_estaciones()
print(f"Estaciones usadas: {len(mens)}  (precipitación: {fichas.usada_precip.sum()}, "
      f"temperatura: {fichas.usada_temp.sum()}, p90: {fichas.usada_p90.sum()})")

Estaciones candidatas: 1240


  200/1240


  400/1240


  600/1240


  800/1240


  1000/1240


  1200/1240


Estaciones usadas: 377  (precipitación: 377, temperatura: 57, p90: 55)


### Estaciones que cumplen completitud según la altitud máxima

In [7]:
def conteos_altitud(fichas: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for alt in ALTITUDES_REPORTE:
        f = fichas[fichas["altitud"] <= alt]
        for d, g in f.groupby("departamento"):
            t = g["completitud_temp"] >= COMPLETITUD_MIN
            filas.append({"altitud_max": alt, "departamento": d,
                          "precip": int((g["completitud_precip"] >= COMPLETITUD_MIN).sum()),
                          "temp": int(t.sum()),
                          "p90": int((t & (g["anios_base_tmax"] >= ANIOS_BASE_MIN)).sum())})
    t = pd.DataFrame(filas).pivot_table(index="departamento", columns="altitud_max",
                                        values=["precip", "temp", "p90"], aggfunc="sum")
    t.columns = [f"{v}_{a}m" for v, a in t.columns]
    return t.reset_index()


alt = conteos_altitud(fichas)
alt

,departamento,p90_1000m,p90_1500m,precip_1000m,precip_1500m,temp_1000m,temp_1500m
0,Antioquia,5,12,61,83,6,13
1,Caquetá,4,4,15,15,4,4
2,Casanare,5,5,28,29,5,5
3,Cesar,4,4,54,55,4,4
4,Córdoba,12,12,54,54,13,13
5,Huila,3,7,40,68,3,9
6,Meta,6,6,49,52,6,6
7,Norte de Santander,4,6,29,39,4,6
8,Tolima,12,12,47,59,12,12


## 7. Agregación

### Por departamento
- **Conteos y precipitación:** promedio simple de las estaciones válidas del mes. Se guarda cuántas estaciones
  aportaron (`n_estaciones_temp`, `n_estaciones_p90`, `n_estaciones_lluvia`).
- **`tmax_media` y `tmin_media`:** promedio de **anomalías**. Si una estación sale o entra, el promedio simple puede
  saltar aunque el clima no cambie (por ejemplo, si deja de reportar la estación más caliente). Para evitarlo:
  1. a cada estación se le resta su propia media por mes calendario (2013-2026) → anomalía;
  2. se promedian las anomalías de las estaciones válidas del mes;
  3. se suma la climatología fija del departamento (promedio de las medias de sus estaciones).

  Así el resultado sigue en °C y no tiene saltos artificiales.

### Por producto
Promedio ponderado de los departamentos con dato en el mes, renormalizando los pesos:
`arroz_x = Σ (peso_d × x_d) / Σ peso_d`. Si los departamentos con dato suman menos del 50 % del peso del producto,
el mes queda `NaN` (columnas `peso_cubierto_*`).

### Tabla del modelo
Una fila por mes con `fecha`, `anio`, `mes` y las columnas `arroz_*` y `res_*`. No incluye THI (el IDEAM casi no tiene
humedad diaria completa en estas estaciones).

In [8]:
def por_departamento(fichas: pd.DataFrame, mens: dict[str, pd.DataFrame]) -> pd.DataFrame:
    largo = pd.concat([m.rename_axis("fecha").reset_index().assign(codigo=cod) for cod, m in mens.items()],
                      ignore_index=True)
    largo = largo.merge(fichas[["codigo", "departamento"]], on="codigo")
    for c in VARIABLES:
        if c not in largo:
            largo[c] = np.nan
    largo["mes"] = largo["fecha"].dt.month

    # anomalía de cada estación respecto a su propia media por mes calendario
    for v in ("tmax_media", "tmin_media"):
        clim = largo.groupby(["codigo", "mes"])[v].transform("mean")
        largo[f"{v}_anom"] = largo[v] - clim
        largo[f"{v}_clim"] = clim

    g = largo.groupby(["departamento", "fecha"])
    dep = g[[v for v in VARIABLES if v not in ("tmax_media", "tmin_media")]].mean()
    for v in ("tmax_media", "tmin_media"):
        clim_dep = (largo.drop_duplicates(["codigo", "mes"])
                    .groupby(["departamento", "mes"])[f"{v}_clim"].mean())
        anom = g[f"{v}_anom"].mean()
        idx = anom.index.to_frame(index=False)
        idx["mes"] = idx["fecha"].dt.month
        dep[v] = anom.to_numpy() + clim_dep.reindex(
            pd.MultiIndex.from_frame(idx[["departamento", "mes"]])).to_numpy()
    dep["n_estaciones_temp"] = g["tmax_media"].count()
    dep["n_estaciones_p90"] = g["dias_calor_p90"].count()
    dep["n_estaciones_lluvia"] = g["precip_total_mm"].count()
    dep["dias_validos_temp"] = g["dias_validos_temp"].mean()
    dep["dias_validos_lluvia"] = g["dias_validos_lluvia"].mean()
    dep["frac_est_reescaladas_temp"] = g["reescalado_temp"].apply(lambda s: s.astype(float).mean())
    dep["frac_est_reescaladas_lluvia"] = g["reescalado_lluvia"].apply(lambda s: s.astype(float).mean())
    dep = dep.reset_index()

    # malla completa departamento x mes (los meses sin ninguna estación quedan NaN)
    deptos = sorted({d for p in PESOS.values() for d in p})
    malla = pd.MultiIndex.from_product([deptos, MESES], names=["departamento", "fecha"]).to_frame(index=False)
    dep = malla.merge(dep, on=["departamento", "fecha"], how="left")
    for c in ("n_estaciones_temp", "n_estaciones_p90", "n_estaciones_lluvia"):
        dep[c] = dep[c].fillna(0).astype(int)
    otras = [c for c in dep.columns if c not in VARIABLES + ["departamento", "fecha"]]
    return dep[["departamento", "fecha"] + VARIABLES + otras]


def por_producto(dep: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for prod, pesos in PESOS.items():
        d = dep[dep["departamento"].isin(pesos)].copy()
        d["peso"] = d["departamento"].map(pesos)
        total = sum(pesos.values())
        for fecha, gr in d.groupby("fecha"):
            fila = {"producto": prod, "fecha": fecha}
            for v in VARIABLES:
                ok = gr[v].notna()
                cubierto = gr.loc[ok, "peso"].sum() / total
                fila[v] = (np.average(gr.loc[ok, v], weights=gr.loc[ok, "peso"])
                           if cubierto >= PESO_MIN_PRODUCTO else np.nan)
                if v in ("tmax_media", "dias_calor_p90", "precip_total_mm"):
                    fila[f"peso_cubierto_{v}"] = round(cubierto, 3)
            fila["n_departamentos_temp"] = int(gr["tmax_media"].notna().sum())
            fila["n_departamentos_lluvia"] = int(gr["precip_total_mm"].notna().sum())
            filas.append(fila)
    return pd.DataFrame(filas)


def tabla_modelo(prod: pd.DataFrame) -> pd.DataFrame:
    anchas = [prod[prod["producto"] == p].set_index("fecha")[cols].add_prefix(f"{p}_")
              for p, cols in VARIABLES_MODELO.items()]
    t = pd.concat(anchas, axis=1).reindex(MESES).rename_axis("fecha").reset_index()
    t.insert(1, "anio", t["fecha"].dt.year)
    t.insert(2, "mes", t["fecha"].dt.month)
    return t


dep = por_departamento(fichas, mens)
prod = por_producto(dep)
tm = tabla_modelo(prod)
for df in (dep, prod):
    df.insert(2, "anio", df["fecha"].dt.year)
    df.insert(3, "mes", df["fecha"].dt.month)
print(f"tabla_modelo: {tm.shape[0]} meses x {tm.shape[1]} columnas")

tabla_modelo: 166 meses x 16 columnas


## 8. Ejemplo: los dos indicadores de calor en una estación real

Estación con temperatura y p90 de Tolima con mayor completitud. Se muestran sus 12 umbrales p90 y, para tres meses de
marzo, la Tmax media, los días por encima del p90 y los días ≥ 35 °C. En la estación Albania (Mariquita, 788 m), el
umbral de marzo es 30,2 °C: marzo de 2016 (El Niño fuerte) tuvo 9 días por encima y marzo de 2018 (La Niña) solo 2,
mientras que `dias_tmax_ge35` da 0 en los tres, porque allí la Tmax medida nunca llega a 35 °C.

In [9]:
ej = (fichas[(fichas.departamento == "Tolima") & fichas.usada_p90]
      .sort_values("completitud_temp", ascending=False).iloc[0])
print(f"{ej.nombre} ({ej.municipio}), {ej.altitud:.0f} m")
t_ej, _ = diario_temp(int(ej.codigo))
p90_ej = umbrales_p90(t_ej)
print("Umbral p90 de Tmax por mes (°C):", p90_ej.round(1).to_dict())

filas = []
for mes in ["2015-03", "2016-03", "2018-03"]:
    m = t_ej.loc[mes, "tmax"].dropna()
    filas.append({"mes": mes, "dias_con_dato": len(m), "tmax_media": round(m.mean(), 2),
                  f"dias > p90 ({p90_ej[3]:.1f} °C)": int((m > p90_ej[3]).sum()),
                  "dias >= 35 °C": int((m >= UMBRAL_TMAX_ARROZ).sum())})
pd.DataFrame(filas)

ALBANIA - AUT [23025040] (Mariquita), 788 m
Umbral p90 de Tmax por mes (°C): {1: 30.0, 2: 30.2, 3: 30.2, 4: 30.0, 5: 29.8, 6: 30.0, 7: 31.2, 8: 31.6, 9: 31.4, 10: 29.6, 11: 28.6, 12: 29.4}


,mes,dias_con_dato,tmax_media,dias > p90 (30.2 °C),dias >= 35 °C
0,2015-03,31,28.66,7,0
1,2016-03,31,29.40,9,0
2,2018-03,31,27.86,2,0


## 9. Salidas

En `salidas/`:
- `clima_ideam_mensual.xlsx`, con las hojas `tabla_modelo`, `por_producto`, `por_departamento`, `estaciones_usadas`
  y `calidad`.
- `clima_ideam_tabla_modelo.csv`.
- `reporte_ideam.md`: resumen para la metodología (estaciones, completitud, calidad, pesos y cobertura).

In [10]:
def tabla_estaciones(fichas: pd.DataFrame) -> pd.DataFrame:
    e = fichas[fichas["usada_precip"] | fichas["usada_temp"]].copy()
    e["variables_disponibles"] = e.apply(
        lambda r: ", ".join(n for n, u in [("precipitacion", r.usada_precip), ("tmax/tmin", r.usada_temp),
                                           ("dias_calor_p90", r.usada_p90)] if u), axis=1)
    e["productos"] = e["departamento"].map(lambda d: ", ".join(p for p, ps in PESOS.items() if d in ps))
    for c in ("completitud_precip", "completitud_temp"):
        e[c] = (100 * e[c]).round(1)
    e["anios_base_tmax"] = e["anios_base_tmax"].round(1)
    cols = ["codigo", "nombre", "departamento", "municipio", "latitud", "longitud", "altitud",
            "categoria", "tecnologia", "estado", "productos", "variables_disponibles",
            "completitud_precip", "completitud_temp", "anios_base_tmax"]
    return e[cols].sort_values(["departamento", "codigo"])


def _md(df: pd.DataFrame) -> str:
    lineas = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)]
    for _, r in df.iterrows():
        lineas.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |")
    return "\n".join(lineas)


def r2(df):
    """Redondea a 2 decimales solo las columnas numéricas."""
    return df.round({c: 2 for c in df.select_dtypes("number").columns})


est = tabla_estaciones(fichas)
calidad_usadas = calidad[calidad["codigo"].isin(est["codigo"])]
resumen = (est.groupby("departamento")
           .agg(estaciones=("codigo", "size"),
                con_precip=("variables_disponibles", lambda s: int(s.str.contains("precipitacion").sum())),
                con_temp=("variables_disponibles", lambda s: int(s.str.contains("tmax").sum())),
                con_p90=("variables_disponibles", lambda s: int(s.str.contains("p90").sum())),
                completitud_precip_media=("completitud_precip", lambda s: round(s.mean(), 1)),
                completitud_temp_media=("completitud_temp", lambda s: round(s.mean(), 1)))
           .reset_index())
resumen

,departamento,estaciones,con_precip,con_temp,con_p90,completitud_precip_media,completitud_temp_media
0,Antioquia,61,61,6,5,94.2,72.0
1,Caquetá,15,15,4,4,86.1,71.7
2,Casanare,28,28,5,5,90.6,79.6
3,Cesar,54,54,4,4,92.1,58.7
4,Córdoba,54,54,13,12,91.7,85.0
5,Huila,40,40,3,3,92.4,72.1
6,Meta,49,49,6,6,93.1,73.1
7,Norte de Santander,29,29,4,4,94.2,64.8
8,Tolima,47,47,12,12,93.3,82.4


In [11]:
cob = pd.DataFrame({"columna": [c for c in tm.columns if c not in ("fecha", "anio", "mes")]})
cob["meses_con_dato"] = cob["columna"].map(lambda c: int(tm[c].notna().sum()))
cob["meses_nan"] = len(tm) - cob["meses_con_dato"]
cob["pct_cobertura"] = (100 * cob["meses_con_dato"] / len(tm)).round(1)

nan_dep = (dep.groupby("departamento")
           .agg(meses_nan_tmax=("tmax_media", lambda s: int(s.isna().sum())),
                meses_nan_p90=("dias_calor_p90", lambda s: int(s.isna().sum())),
                meses_nan_precip=("precip_total_mm", lambda s: int(s.isna().sum())),
                min_est_temp=("n_estaciones_temp", lambda s: int(s[s > 0].min()) if (s > 0).any() else 0),
                min_est_lluvia=("n_estaciones_lluvia", lambda s: int(s[s > 0].min()) if (s > 0).any() else 0))
           .reset_index())
cols_q = [c for c in calidad_usadas.columns if c.startswith(("precip_desc", "temp_desc", "tmax_desc", "precip_dias", "temp_dias"))]
tot_q = calidad_usadas[cols_q].sum().astype(int).rename("total").reset_index().rename(columns={"index": "regla"})
pesos_df = pd.DataFrame([{"producto": p, "departamento": d, "peso": w,
                          "participacion_%": round(100 * w / sum(ps.values()), 1)}
                         for p, ps in PESOS.items() for d, w in ps.items()])

reporte = f"""# Variables climáticas mensuales desde estaciones del IDEAM

Generado por `clima_ideam.ipynb`. Periodo de estudio: {FECHA_INICIO[:7]} a {FECHA_FIN[:7]} ({len(tm)} meses).
El notebook contiene la explicación detallada de cada paso e indicador.

## Fuentes
- Precipitación diaria: `PrecipitacionNacionalDiaria.zip` (IDEAM, bart.ideam.gov.co, vía https; el http no respondió).
- Temperatura diaria: `TemperaturaNacionalDiaria.zip` (IDEAM).
- Catálogo Nacional de Estaciones (CNE): datos.gov.co, conjunto `hp9r-jxuu`.
- Solo se usa información del IDEAM; no hay relleno con fuentes satelitales.

## Variables y etiquetas (verificadas con los datos)
- `PTPM_CON_INTER`: precipitación total diaria (lectura de las 07:00, asignada a la fecha de lectura).
- `TSSM_MX_D` / `TSSM_MN_D`: máximo / mínimo de las 3 lecturas diarias de termómetro seco (07, 13 y 19 h).
  Los masivos no incluyen TMX_CON / TMN_CON. La Tmax usada subestima la real y la Tmin la sobreestima;
  `dias_tmax_ge35` es una cota inferior y `dias_calor_p90` es robusto al sesgo (indicador principal de calor).
- `dias_calor_p90`: el 8,9 % de los días del periodo base supera el p90 (empates por resolución de 0,2 °C);
  la frecuencia crece de 6,6 % (1991-2002) a 13,5 % (2013-2020) por la tendencia de calentamiento.

## Criterios de selección
- Altitud máxima {ALTITUD_MAX} m s. n. m.; completitud mínima {COMPLETITUD_MIN:.0%} por variable en el periodo de estudio.
- `dias_calor_p90`: al menos {ANIOS_BASE_MIN} años de Tmax entre {BASE_INICIO[:4]} y {int(FECHA_INICIO[:4]) - 1};
  percentil sobre {BASE_INICIO[:4]}-{BASE_FIN[:4]} por estación y mes calendario.
- Tecnologías excluidas: {TECNOLOGIAS_EXCLUIDAS or 'ninguna'} (las series `_CON` son convencionales).

### Estaciones que cumplen completitud según altitud máxima

{_md(alt)}

### Estaciones usadas por departamento (altitud <= {ALTITUD_MAX} m)

{_md(resumen)}

## Control de calidad diario
Precipitación < {PRECIP_MIN} o > {PRECIP_MAX} mm/día; Tmax < {TMAX_MIN} o > {TMAX_MAX} °C; Tmin > Tmax (ambas);
fechas repetidas. Un mes de una estación queda NaN con menos del {DIAS_MES_MIN:.0%} de días con dato.

{_md(tot_q)}

## Indicadores
tmax_media, tmin_media, dias_calor_p90, dias_tmax_ge35, precip_total_mm, dias_lluvia (>= {UMBRAL_LLUVIA_MM} mm),
dias_lluvia_fuerte (>= {UMBRAL_LLUVIA_FUERTE_MM} mm), max_racha_seca. Los conteos se reescalan al largo del mes;
`max_racha_seca` y `precip_total_mm` no. Sin THI. No hay indicador de frío extremo ni de lluvia relativa (percentil).

## Agregación
- Departamento: promedio de estaciones válidas; tmax/tmin por anomalías + climatología fija del departamento.
- Producto: promedio ponderado renormalizado; NaN si el peso cubierto es menor al {PESO_MIN_PRODUCTO:.0%}.

### Pesos (fijos)
- Arroz: producción anual (miles de t), promedio 2013-2025, DANE-Fedearroz ENAM (Norte de Santander 2018-2025).
- Res: inventario bovino (millones de cabezas), promedio censos ICA 2018 y 2020-2025.

{_md(pesos_df)}

### Meses sin dato por departamento

{_md(nan_dep)}

## Cobertura de la tabla del modelo

{_md(cob)}

Los NaN restantes no se rellenan aquí; su tratamiento se decide en la etapa de modelado.
"""

CARPETA_SALIDA.mkdir(exist_ok=True)
hojas = {"tabla_modelo": r2(tm), "por_producto": r2(prod), "por_departamento": r2(dep),
         "estaciones_usadas": est, "calidad": calidad_usadas}
xlsx = CARPETA_SALIDA / "clima_ideam_mensual.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    for nombre, df in hojas.items():
        df2 = df.copy()
        if "fecha" in df2:
            df2["fecha"] = df2["fecha"].dt.date
        df2.to_excel(w, sheet_name=nombre, index=False)
r2(tm).to_csv(CARPETA_SALIDA / "clima_ideam_tabla_modelo.csv", index=False, encoding="utf-8")
(CARPETA_SALIDA / "reporte_ideam.md").write_text(reporte, encoding="utf-8")
print(f"Listo: {xlsx}, clima_ideam_tabla_modelo.csv y reporte_ideam.md")

Listo: salidas\clima_ideam_mensual.xlsx, clima_ideam_tabla_modelo.csv y reporte_ideam.md


### Vista previa y cobertura

In [12]:
r2(tm.head(12))

,fecha,anio,mes,arroz_tmax_media,arroz_dias_calor_p90,arroz_dias_tmax_ge35,arroz_precip_total_mm,arroz_dias_lluvia,arroz_dias_lluvia_fuerte,arroz_max_racha_seca,res_tmax_media,res_dias_calor_p90,res_precip_total_mm,res_dias_lluvia,res_dias_lluvia_fuerte,res_max_racha_seca
0,2013-01-01,2013,1,31.42,3.00,1.04,34.48,2.54,0.66,21.76,31.75,4.68,20.04,1.44,0.32,25.57
1,2013-02-01,2013,2,30.97,0.53,1.00,101.87,6.81,1.79,12.95,31.26,1.32,97.25,5.88,1.77,14.56
2,2013-03-01,2013,3,30.77,2.54,1.55,161.44,9.91,3.03,7.65,30.59,0.89,197.91,11.39,3.76,7.34
3,2013-04-01,2013,4,30.91,6.44,1.85,250.85,10.67,4.54,9.30,30.65,3.90,264.73,11.32,4.70,8.14
4,2013-05-01,2013,5,29.02,1.17,0.05,331.73,15.99,5.91,4.96,29.42,2.11,363.04,16.17,6.54,4.97
5,2013-06-01,2013,6,29.88,4.11,1.19,192.69,10.59,3.27,8.88,29.65,3.44,278.86,14.40,5.07,5.42
6,2013-07-01,2013,7,29.52,2.76,1.09,213.64,11.28,3.84,8.38,29.37,2.70,283.51,13.99,5.10,6.44
7,2013-08-01,2013,8,29.88,0.71,1.48,186.29,10.47,3.31,8.46,29.30,1.02,291.23,14.79,5.34,4.97
8,2013-09-01,2013,9,31.03,2.71,3.54,170.80,10.08,3.04,7.78,29.95,2.28,257.34,12.96,4.67,5.50
9,2013-10-01,2013,10,30.80,4.46,1.85,184.19,10.63,3.28,7.86,30.22,3.44,248.51,12.54,4.46,6.06


In [13]:
cob

,columna,meses_con_dato,meses_nan,pct_cobertura
0,arroz_tmax_media,163,3,98.2
1,arroz_dias_calor_p90,163,3,98.2
2,arroz_dias_tmax_ge35,163,3,98.2
3,arroz_precip_total_mm,163,3,98.2
4,arroz_dias_lluvia,163,3,98.2
5,arroz_dias_lluvia_fuerte,163,3,98.2
6,arroz_max_racha_seca,163,3,98.2
7,res_tmax_media,163,3,98.2
8,res_dias_calor_p90,163,3,98.2
9,res_precip_total_mm,163,3,98.2
